# Stochastic Volatility Generator

Generate price paths with stochastic volatility models including Heston and SABR.
These models capture the empirical observation that volatility itself is random and
often negatively correlated with price movements (the leverage effect).

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from synforecast.generators import StochasticVolatilityGenerator

## 1. Heston Model (1 Year Daily)

The Heston model features mean-reverting stochastic variance with a leverage effect
(negative correlation between price and volatility).

In [ ]:
params = {
    "min_length": 252,
    "max_length": 252,
    "freq": "D",
    "model": "heston",
    "initial_price": 100.0,
    "initial_vol": 0.04,
    "drift": 0.05,
    "mean_vol": 0.04,
    "vol_mean_reversion": 2.0,
    "vol_of_vol": 0.3,
    "correlation": -0.7,
    "seed": 42,
}

generator = StochasticVolatilityGenerator(engine="polars", **params)
df = generator.generate(n_series=3)

print(f"Generated {df['unique_id'].n_unique()} price paths")

stats = df.group_by("unique_id").agg(
    [
        pl.col("y").first().alias("start_price"),
        pl.col("y").last().alias("end_price"),
        pl.col("y").min().alias("min_price"),
        pl.col("y").max().alias("max_price"),
    ]
)
stats

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df["unique_id"].unique().to_list():
    series = df.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Price")
ax.set_title("Heston Model - Price Paths")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Price and Volatility Paths

Generate both the price and volatility paths to observe their joint dynamics.

In [ ]:
prices, vols, ids = generator.generate_with_volatility(n_series=1)
print(f"Price range: [{prices.min():.2f}, {prices.max():.2f}]")
print(f"Volatility range: [{vols.min():.3f}, {vols.max():.3f}]")
print(f"Mean volatility: {vols.mean():.3f} ({vols.mean() * np.sqrt(252) * 100:.1f}% annualized)")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(prices, alpha=0.8)
axes[0].set_ylabel("Price")
axes[0].set_title("Heston Model - Price and Volatility Paths")
axes[1].plot(vols, alpha=0.8, color="tab:orange")
axes[1].set_xlabel("Time Step")
axes[1].set_ylabel("Variance")
plt.tight_layout()
plt.show()

## 3. Leverage Effect (Price-Vol Correlation)

Verify that returns and volatility changes are negatively correlated (leverage effect).

In [ ]:
returns = np.diff(np.log(prices))
vol_changes = np.diff(vols)
corr = np.corrcoef(returns, vol_changes)[0, 1]
print(f"Return-VolChange correlation: {corr:.3f}")
print("(Negative = leverage effect: vol rises when prices fall)")

## 4. SABR Model

The SABR (Stochastic Alpha Beta Rho) model is widely used for interest rate derivatives
and allows a CEV exponent (beta) to control the volatility smile shape.

In [ ]:
sabr_gen = StochasticVolatilityGenerator(engine="polars", 
    **{
        "min_length": 252,
        "max_length": 252,
        "freq": "D",
        "model": "sabr",
        "initial_price": 100.0,
        "beta": 0.5,
        "correlation": -0.3,
        "seed": 42,
    }
)
sabr_df = sabr_gen.generate(n_series=1)
print(f"SABR price range: [{sabr_df['y'].min():.2f}, {sabr_df['y'].max():.2f}]")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in sabr_df["unique_id"].unique().to_list():
    series = sabr_df.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Price")
ax.set_title("SABR Model - Price Path")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Model Information

Inspect the full model parameters.

In [ ]:
info = generator.get_model_info()
for key, value in info.items():
    print(f"{key}: {value}")

## 6. Implied Volatility Smile (SABR)

Compute the implied volatility smile from the SABR model for various strike prices.

In [ ]:
strikes = np.array([80, 90, 95, 100, 105, 110, 120])
impl_vols = sabr_gen.implied_volatility_smile(strikes, maturity=1.0)

print("Strike | Implied Vol")
print("-" * 25)
for k, iv in zip(strikes, impl_vols):
    print(f"  {k:3d}  |   {iv * 100:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(strikes, impl_vols * 100, marker="o", linewidth=2)
ax.set_xlabel("Strike Price")
ax.set_ylabel("Implied Volatility (%)")
ax.set_title("SABR Implied Volatility Smile")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Output Types

The generator supports different output types: price, returns, and volatility.

In [ ]:
for output_type in ["price", "returns", "volatility"]:
    gen = StochasticVolatilityGenerator(engine="polars", 
        **{
            "min_length": 100,
            "max_length": 100,
            "freq": "D",
            "output_type": output_type,
            "seed": 42,
        }
    )
    out_df = gen.generate(n_series=1)
    vals = out_df["y"].to_numpy()
    print(f"{output_type:12s}: mean={vals.mean():.4f}, std={vals.std():.4f}")